In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("Stacking Ensemble")

In [ ]:
# Load and clean data
dataset = pd.read_csv('dataset.csv')
cleaned_dataset = dataset.dropna()

# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Split BEFORE vectorizing to avoid data leakage
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

# TF-IDF - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
tfidf_cleaned = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_tfidf_cleaned = tfidf_cleaned.fit_transform(X_train_cleaned)  # Fit ONLY on train
X_test_tfidf_cleaned = tfidf_cleaned.transform(X_test_cleaned)  # Transform test

In [ ]:
# Base learners (trained on the TF-IDF features)
base_lightgbm = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    class_weight="balanced",  # handles imbalance (is_unbalance removed - binary-only & conflicts with class_weight)
    reg_alpha=0.1,   # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    learning_rate=0.08081298097796712,
    n_estimators=367,
    max_depth=20,
    random_state=42,
    verbose=-1
)

base_logreg = LogisticRegression(max_iter=1000, class_weight='balanced', solver='lbfgs')

# Meta-learner (trained on the base models' out-of-fold predicted probabilities)
meta_logreg = LogisticRegression(max_iter=1000, solver='lbfgs')

# Stacking: LightGBM + LogisticRegression as base models, LogisticRegression as meta-learner
stacking_model = StackingClassifier(
    estimators=[
        ('base_lightgbm', base_lightgbm),
        ('base_logreg', base_logreg)
    ],
    final_estimator=meta_logreg,
    cv=5
)

# Train, evaluate, and log to MLflow
with mlflow.start_run():
    mlflow.set_tag("mlflow.runName", "StackingClassifier_LGBM_LogReg_LogRegMeta")
    mlflow.set_tag("model_type", "StackingClassifier")
    mlflow.log_param("vectorizer_type", "TfidfVectorizer")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("vectorizer_max_features", max_features)
    mlflow.log_param("base_estimators", "base_lightgbm, base_logreg")
    mlflow.log_param("meta_learner", "LogisticRegression")
    mlflow.log_param("cv", 5)

    # Train the stacking model
    stacking_model.fit(X_train_tfidf_cleaned, y_train_cleaned)

    # Predict on the test set
    y_pred = stacking_model.predict(X_test_tfidf_cleaned)

    # Log accuracy
    accuracy = accuracy_score(y_test_cleaned, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Log classification report metrics
    report = classification_report(y_test_cleaned, y_pred, output_dict=True)
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # Log the final stacking model (single artifact - fine for DagShub free tier)
    mlflow.sklearn.log_model(stacking_model, "stacking_model")

    print(f"Accuracy: {accuracy:.4f}\n")
    print(classification_report(y_test_cleaned, y_pred))